# Indic Audio Filtering Pipeline — Dataset Setup

This notebook prepares the IndicVoices dataset in **Google Colab + Google Drive** using the provided setup script.

It directly supports the assignment requirement to use a large multilingual Indic speech dataset and generate manifests for scalable processing.  
Assignment reference: `Audio Filtering Pipeline for Indic Speech` and provided `setup_dataset.py`. fileciteturn2file0 fileciteturn2file1


In [ ]:
%%capture
!pip install -q polars tqdm huggingface_hub librosa soundfile matplotlib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/indic_audio_filter"
DATA_DIR = f"{DRIVE_DIR}/data"
PROJECT_DIR = f"{DRIVE_DIR}/project"

import os, pathlib
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(PROJECT_DIR, exist_ok=True)

print("Drive root:", DRIVE_DIR)
!df -h /content


## Upload the provided `setup_dataset.py`

If needed, upload the provided script from the assignment bundle. It downloads a subset of the gated IndicVoices dataset and writes:
- `hf/`
- `audios/`
- `manifests/`
- `setup.log` fileciteturn2file1


In [ ]:
from google.colab import files
uploaded = files.upload()  # upload setup_dataset.py if needed


In [ ]:
import os, shutil, glob
for name in glob.glob("/content/*.py"):
    if os.path.basename(name).startswith("setup_dataset"):
        shutil.copy(name, f"{PROJECT_DIR}/setup_dataset.py")
        print("Copied:", name)


In [ ]:
!huggingface-cli login


In [ ]:
!python "{PROJECT_DIR}/setup_dataset.py" "{DATA_DIR}"


In [ ]:
import polars as pl, glob, os
manifests = sorted(glob.glob(f"{DATA_DIR}/manifests/**/*.jsonl", recursive=True))
print("Manifest files:", len(manifests))
langs = {}
for mf in manifests:
    try:
        df = pl.read_ndjson(mf)
        lang = os.path.basename(os.path.dirname(mf)).replace("_manifests","")
        langs[lang] = langs.get(lang, 0) + len(df)
    except Exception as e:
        print("Failed:", mf, e)
print("Languages:", sorted(langs.keys())[:15])
print("Sample counts by language (first 10):")
for k in list(sorted(langs.keys()))[:10]:
    print(k, langs[k])


In [ ]:
import matplotlib.pyplot as plt
durations = []
for mf in manifests[:8]:
    df = pl.read_ndjson(mf)
    for c in ['duration', 'duration_sec']:
        if c in df.columns:
            durations.extend(df[c].to_list())
if durations:
    plt.figure(figsize=(10,4))
    plt.hist(durations, bins=60)
    plt.axvline(0.5, linestyle='--')
    plt.axvline(30.0, linestyle='--')
    plt.title("Manifest duration distribution (sample of shards)")
    plt.xlabel("Duration (sec)")
    plt.ylabel("Count")
    plt.tight_layout()
    os.makedirs(f"{DRIVE_DIR}/reports/plots", exist_ok=True)
    plt.savefig(f"{DRIVE_DIR}/reports/plots/raw_duration_distribution.png", dpi=150)
    plt.show()
else:
    print("Duration field not present in sampled manifests.")
